# Jack The Learner - Colab Training

**A humanoid robot brain that understands physics - built on transformers.**

## Setup
1. **Runtime → Change runtime type → T4 GPU** (or A100 for vision)
2. Run cells in order
3. Checkpoints save to Google Drive automatically

## Training Pipeline
```
Phase 0 (1-2 hrs)     Phase 1 (4-12 hrs)    Phase 2 (2-4 hrs)
Learn Physics    →    Learn Walking     →    Learn from Demos
MathReasoner          + WorldModel            ALL components
from SymPy            + HAC skills            refined
```

---

## 1. Setup Environment

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/JackTheLearner/checkpoints', exist_ok=True)
print('Google Drive mounted')

In [ ]:
# Install dependencies
!pip install -q torch torchvision
!pip install -q gymnasium mujoco
!pip install -q sympy tqdm tensorboard
print('Dependencies installed')

In [ ]:
# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU! Runtime > Change runtime type > T4 GPU')

In [ ]:
# Clone repository and link checkpoints to Drive
%cd /content
!rm -rf JackTheLearner 2>/dev/null
!git clone https://github.com/JannoLouwrens/JackTheLearner.git
%cd JackTheLearner

# IMPORTANT: Symlink checkpoints to Drive so every save goes directly to Drive
!rm -rf checkpoints 2>/dev/null
!ln -s /content/drive/MyDrive/JackTheLearner/checkpoints checkpoints

print('Checkpoints folder linked to Drive - saves are instant!')
!ls -la checkpoints/

---
## 2. Phase 0: Learn Physics (1-2 hours)

Neural network learns F=ma, torque, energy from SymPy ground truth.

In [ ]:
# Quick test (5 min)
!python Phase0_Physics.py --samples 1000 --epochs 5

In [ ]:
# Full Phase 0 (1-2 hours)
# Checkpoints save directly to Drive via symlink!
!python Phase0_Physics.py --samples 100000 --epochs 50
print('Phase 0 complete!')

---
## 3. Phase 1: Learn Walking (4-12 hours)

RL training in MuJoCo. Auto-resumes from `rl_latest.pt` if exists.

In [ ]:
# Check existing checkpoints (already on Drive via symlink)
!ls -la checkpoints/

In [ ]:
# Phase 1: RL Walking
# Auto-resumes if rl_latest.pt exists! Saves directly to Drive!

!python Phase1_Locomotion.py \
    --phase0-checkpoint checkpoints/phase0_best.pt \
    --epochs 500

print('Phase 1 complete!')

---
## 4. Phase 2: Learn from Demos (2-4 hours)

Imitation learning for natural movement.

In [ ]:
# Check Phase 1 checkpoint exists before Phase 2
!ls -la checkpoints/rl_best.pt

In [ ]:
# Phase 2: Imitation Learning
# Saves directly to Drive!

!python Phase2_Imitation.py \
    --checkpoint-in checkpoints/rl_best.pt \
    --epochs 100

print('Phase 2 complete!')

---
## 5. Results

In [ ]:
# List all checkpoints (on Drive)
!ls -lh checkpoints/

---
## Checkpoint Names

| Phase | Checkpoint | Description |
|-------|------------|-------------|
| 0 | `phase0_best.pt` | MathReasoner with physics |
| 1 | `rl_latest.pt` | Latest RL checkpoint (auto-resume) |
| 1 | `rl_best.pt` | Best RL checkpoint |
| 2 | `phase2_best.pt` | Final trained brain |

## How Saves Work
- `checkpoints/` is symlinked to Google Drive
- **Every save goes directly to Drive** - no manual copying needed
- If Colab crashes, your latest checkpoint is safe on Drive

## Resume After Disconnect
1. Re-run setup cells (1-5)
2. The symlink reconnects to your Drive checkpoints
3. Training auto-resumes from where it left off

---
**Author:** Janno Louwrens